# Diffusion Language Models and Parallel Generation

> An autoregressive model needs 200 sequential forward passes to write 200 tokens. This serial dependency is built into next-token prediction and cannot be removed by idle GPU compute.
>
> Image diffusion generates an entire image together over dozens of denoising steps. Text is harder because pixels are continuous numbers, while tokens are discrete categories with no natural “small amount of noise.”
>
> This appendix replaces noise with `[MASK]`, transfers diffusion ideas to language, trains a toy model, and measures its quality–step trade-off against autoregressive generation.


## 0. The Serial Bottleneck of Autoregression

Generating length $L$ autoregressively requires $L$ ordered forward passes because token $i$ depends on the preceding $i-1$. Speculative decoding validates several proposals per target pass on average, but remains within this serial framework.

**Non-Autoregressive Generation (NAR)** produces all tokens in one or a small number of forward passes rather than strictly left to right. In plain language, it writes a full draft and revises it instead of typing one token at a time.

One-pass NAR usually has poor language quality because positions make individually plausible but mutually inconsistent choices. A small hand example demonstrates the failure.


In [ ]:
import math
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device)


In [ ]:
# Toy corpus: restaurant reviews contain only two fixed collocations
# "tasty and affordable" and "bad and expensive": the two blanks are correlated

p_slot1 = {"tasty": 0.5, "bad": 0.5}
p_slot2_given = {
    "tasty": {"affordable": 0.9, "expensive": 0.1},  # after tasty, affordable is likely
    "bad": {"expensive": 0.9, "affordable": 0.1},   # after bad, expensive is likely
}

# One-shot generation fills both blanks together, so the second uses only its marginal distribution
p_slot2_marginal = {"affordable": 0.5, "expensive": 0.5}
one_shot_ok = (p_slot1["tasty"] * p_slot2_marginal["affordable"]
               + p_slot1["bad"] * p_slot2_marginal["expensive"])

# Left-to-right generation sees the first blank before filling the second
ar_ok = (p_slot1["tasty"] * p_slot2_given["tasty"]["affordable"]
         + p_slot1["bad"] * p_slot2_given["bad"]["expensive"])

print(f"One-shot generation (independent blanks): valid-collocation probability = {one_shot_ok:.0%}")
print(f"Left-to-right generation (first blank first): valid-collocation probability = {ar_ok:.0%}")
print()
print("Key observation: each blank is 90% confident alone, yet independent filling produces a valid result only half the time.")
print("The problem with one-shot generation is not choosing individual words incorrectly, but combining incompatible words.")


In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 3.2))
ax.bar(["one-shot\n(independent slots)", "left-to-right\n(AR)"],
       [one_shot_ok, ar_ok], color=["steelblue", "tomato"], width=0.5)
ax.set_ylim(0, 1)
ax.set_ylabel("probability of a valid phrase")
ax.set_title("Why one-shot generation fails: slots must coordinate")
for i, v in enumerate([one_shot_ok, ar_ok]):
    ax.text(i, v + 0.03, f"{v:.0%}", ha="center")
plt.show()

With only two blanks, joint correctness falls from 90% for coordinated choices to 50% for independent one-step choices. Real sentences contain many dependent positions, so incompatibility compounds.

| Method | Forward passes | Coordination between positions |
|:---|:---|:---|
| Autoregressive | $L$, serial | Later tokens condition on earlier tokens |
| Speculative | roughly $L$/accepted length | Same AR dependency structure |
| One-step NAR | 1 | Positions decide independently |
| Masked Diffusion | adjustable $k$ | Every step rereads global context |

Masked Diffusion accepts that one pass is insufficient. Each round commits high-confidence positions and revisits the rest.


## 1. A Minimal Mental Model of Image Diffusion

A **Diffusion Model** defines a forward noising process and a learned reverse denoising process. Instead of directly learning generation, it learns to clean corrupted examples; generation starts from pure noise and repeatedly cleans it.

- **Forward:** gradually add Gaussian noise according to timestep $t$, from clean data at $t=0$ to near-pure noise at $t=1$.
- **Reverse:** a network receives noisy data and $t$, predicts a cleaner state, and iterates back toward a sample.


In [ ]:
# Demonstrate the forward process on a synthetic 24x24 image as noise gradually covers the pattern
img = np.zeros((24, 24))
img[8:16, 4:20] = 1.0   # horizontal bar
img[4:20, 10:14] = 1.0  # vertical bar, forming a cross

rng = np.random.default_rng(42)
fig, axes = plt.subplots(1, 5, figsize=(12, 2.6))
for ax, t in zip(axes, [0.0, 0.2, 0.5, 1.0, 2.0]):
    noisy = np.clip(img + rng.normal(0, t, img.shape), 0, 1)
    ax.imshow(noisy, cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"noise level t = {t:.1f}")
    ax.axis("off")
plt.suptitle("Forward process: an image is gradually destroyed by noise", y=1.04)
plt.show()

print("Key observation: the reverse process moves from the fully masked state back toward the original sequence.")
print("Every step updates the whole image simultaneously, and the denoising network sees the entire current image.")


Parallelism is inherent here: every pixel update in one step uses global information. For text, removing the causal mask gives every position access to all others. Bidirectional Attention is the foundation of text diffusion.


## 2. Replacing Noise with `[MASK]`

Gaussian noise is defined for continuous pixels but not discrete tokens. **Masked Diffusion** defines noising as replacing tokens with `[MASK]` at some probability and denoising as predicting the original tokens at masked positions.

- **Forward:** independently mask each position with probability $t$; $t=0$ is clean text and $t=1$ is all masks.
- **Reverse:** a bidirectional model outputs a vocabulary distribution at every masked position.


In [ ]:
# Demonstrate the forward process on text; █ represents [MASK]
demo_text = "12+34=046"
rng = np.random.default_rng(42)

fig, axes = plt.subplots(1, 4, figsize=(11, 2.0))
for ax, t in zip(axes, [0.25, 0.5, 0.75, 1.0]):
    shown = [c if rng.random() > t else "█" for c in demo_text]
    for col, ch in enumerate(shown):
        if ch == "█":
            face, edge = "#cccccc", "none"
        else:
            face, edge = "#e8f0fe", "#7a9cc6"
        ax.text(col, 0, ch, ha="center", va="center", fontsize=13,
                bbox=dict(boxstyle="square,pad=0.4", facecolor=face, edgecolor=edge))
    ax.set_xlim(-0.8, len(demo_text) - 0.2)
    ax.set_ylim(-0.6, 0.6)
    ax.axis("off")
    ax.set_title(f"mask rate t = {t:.2f}", fontsize=10)
plt.suptitle("Forward process on text: masking plays the role of noise", y=1.12)
plt.show()

print("Key observation: as t grows, [MASK] gradually covers the sentence; at t=1 all information is gone.")
print("Generation runs this process backward: start from all [MASK] and restore information step by step.")


The objective resembles BERT's MLM but is used differently:

| | BERT MLM | Masked Diffusion |
|:---|:---|:---|
| Training mask ratio | Usually fixed near 15% | Sampled from 0–100% |
| Prediction passes | One | Repeated during sampling |
| Attention | Bidirectional | Bidirectional |

BERT fills one corruption for representation learning. Diffusion turns corruption and repair into a path from all masks to a complete sequence.

Each reverse round predicts all masks, treats the maximum probability as confidence, reveals the top-$k$ positions, and leaves the rest masked. This **confidence-first decoding** postpones hard decisions until more context is available. Other choices are called remasking strategies.


## 3. Hand Calculation: Three-Step Generation

If $m$ masks remain and $s$ steps remain including the current one, reveal $k=\lceil m/s\rceil$ positions. This guarantees completion.

Sort `9 2 5 1`, producing `1 2 5 9` in a seven-character answer region including spaces. At step 1, all seven positions are masked and $k=3$; fixed space positions receive confidence near 0.98 and are revealed first. At step 2, four digit positions remain and $k=2$; the extreme digits 1 and 9 are more certain and are revealed. At step 3, the remaining 2 and 5 become nearly certain from the established context.

The important observation is that confidence at a difficult position may rise from about 0.45 to 0.95 without changing the position itself. Newly revealed tokens provide better context. One-step generation must make every decision when information is least complete.


## 4. Implementing a Mini Masked Diffusion LM

The task sorts eight random digits. A complete example resembles `8 6 5 2 3 0 0 0=0 0 0 2 3 5 6 8`.

- The 15-token answer makes parallel-step savings visible.
- Exact match provides automatic evaluation.
- Every output depends on global input, so bidirectional Attention matters.

The model reuses the Mini-GPT structure but removes the causal mask. We omit a timestep embedding for the toy task; production models usually include one.


In [ ]:
def make_sample(rng, n_nums=8):
    """Generate n_nums random digits from 0-9 and return a 'shuffled=sorted' string."""
    nums = list(rng.integers(0, 10, size=n_nums))
    left = " ".join(str(d) for d in nums)
    right = " ".join(str(d) for d in sorted(nums))
    return f"{left}={right}"

CHARS = sorted(set("0123456789 ="))
char2id = {c: i for i, c in enumerate(CHARS)}
MASK_ID = len(CHARS)           # [MASK] is last in the vocabulary
VOCAB = len(CHARS) + 1

def encode(s):
    return [char2id[c] for c in s]

def decode(ids):
    """Decode IDs into a string, displaying [MASK] as █."""
    table = {i: c for c, i in char2id.items()}
    return "".join(table.get(i, "█") for i in ids)

rng = np.random.default_rng(1)
data = torch.tensor([encode(make_sample(rng)) for _ in range(20000)])

SEQ_LEN = data.shape[1]         # 8 digits + 7 spaces + = + 15 answer characters = 31
ANS_START = 16                  # first position after '=', where the answer begins
ANS_LEN = SEQ_LEN - ANS_START   # 15

print("Example:", decode(data[0].tolist()))
print(f"Sequence length {SEQ_LEN}; answer region {ANS_LEN} positions; vocabulary {VOCAB}, including [MASK]")
print("Key observation: autoregressive decoding needs 15 forward passes for 15 answer positions; now compare diffusion.")


The model differs from Mini-GPT through one `bidirectional` option. When true, it supplies no causal mask and all positions can attend to one another.


In [ ]:
class Block(nn.Module):
    """Standard Transformer Block: Pre-LN + Self-Attention + MLP."""

    def __init__(self, d, n_head):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, n_head, batch_first=True)
        self.ln2 = nn.LayerNorm(d)
        self.mlp = nn.Sequential(
            nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d)
        )

    def forward(self, x, attn_mask):
        h = self.ln1(x)
        a, _ = self.attn(h, h, h, attn_mask=attn_mask)
        x = x + a
        return x + self.mlp(self.ln2(x))


class TinyTransformer(nn.Module):
    """A small Transformer. With bidirectional=True it has no causal mask for diffusion;
    False adds a causal mask for the autoregressive baseline."""

    def __init__(self, d=128, n_layer=3, n_head=4, bidirectional=True):
        super().__init__()
        self.tok = nn.Embedding(VOCAB, d)
        self.pos = nn.Embedding(40, d)
        self.blocks = nn.ModuleList([Block(d, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d)
        self.head = nn.Linear(d, VOCAB)
        self.bidirectional = bidirectional

    def forward(self, idx):
        T = idx.shape[1]
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        if self.bidirectional:
            mask = None
        else:
            # True above the diagonal forbids looking into the future: the causal mask
            mask = torch.triu(
                torch.ones(T, T, dtype=torch.bool, device=idx.device), diagonal=1
            )
        for b in self.blocks:
            x = b(x, mask)
        return self.head(self.ln_f(x))


In [ ]:
torch.manual_seed(42)
diff_model = TinyTransformer(bidirectional=True).to(device)
print("Parameter count:", sum(p.numel() for p in diff_model.parameters()))


Training repeatedly corrupts and restores:

1. Sample a noise level $t\sim U(0,1)$ for each sequence.
2. Independently replace positions with `[MASK]` at probability $t$.
3. Calculate Cross-Entropy only at masked positions.

Compared with BERT MLM, the mask ratio varies across the full range. The original notebook training length is retained; hardware determines elapsed time.


In [ ]:
def train_diffusion(model, steps=3000, bs=128, lr=1e-3):
    """Train a masked diffusion model and return step-loss pairs for plotting."""
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    hist = []
    for step in range(steps):
        batch = data[torch.randint(0, len(data), (bs,))].to(device)
        t = torch.rand(bs, 1, device=device)               # one noise level per sequence
        mask = torch.rand(bs, SEQ_LEN, device=device) < t  # True positions are masked out
        if not mask.any():
            continue                                        # occasionally no position is selected
        x = batch.clone()
        x[mask] = MASK_ID
        logits = model(x)
        loss = F.cross_entropy(logits[mask], batch[mask])   # compute loss only at masked positions
        opt.zero_grad()
        loss.backward()
        opt.step()
        if (step + 1) % 500 == 0:
            print(f"  step {step + 1:5d}  loss {loss.item():.4f}")
            hist.append((step + 1, loss.item()))
    return hist


In [ ]:
t0 = time.time()
diff_hist = train_diffusion(diff_model, steps=3000)
print(f"Training complete in {time.time() - t0:.0f} seconds")


The decoding loop follows the hand calculation: predict every masked position, reveal the top-$k$ by confidence, and leave the remainder for the next round.


In [ ]:
@torch.no_grad()
def diffuse_generate(model, prompt_ids, n_steps, record=False):
    """Denoise for n_steps starting with the entire answer region set to [MASK].
    With record=True, also return a sequence snapshot after every step for visualization."""
    model.eval()
    seq = list(prompt_ids) + [MASK_ID] * ANS_LEN
    masked = list(range(ANS_START, SEQ_LEN))
    snaps = [list(seq)]
    for step_i in range(n_steps):
        ids = torch.tensor([seq], device=device)
        logits = model(ids)[0]
        pos = torch.tensor(masked, device=device)
        probs = F.softmax(logits[pos], dim=-1)
        conf, pick = probs.max(dim=-1)                   # maximum probability = confidence
        k = math.ceil(len(masked) / (n_steps - step_i))  # guarantee completion within n_steps
        top = conf.argsort(descending=True)[:k]          # k highest-confidence positions
        for j in top.tolist():
            seq[masked[j]] = pick[j].item()
        chosen = set(top.tolist())
        masked = [masked[j] for j in range(len(masked)) if j not in chosen]
        snaps.append(list(seq))
    assert not masked, "No [MASK] should remain after decoding"
    return (seq, snaps) if record else seq


In [ ]:
# Fix one test batch for all subsequent evaluations
test_rng = np.random.default_rng(99)
tests = [make_sample(test_rng) for _ in range(200)]

sample = tests[0]
prompt = encode(sample[:ANS_START])
print("Visible input:", sample[:ANS_START])
print("Reference answer:", sample[ANS_START:])
seq, snaps = diffuse_generate(diff_model, prompt, n_steps=5, record=True)
print('Result:', decode(seq)[ANS_START:])
print("Entire sequence correct:", decode(seq) == sample)
print()
for i, snap in enumerate(snaps):
    label = "initial" if i == 0 else f"step {i}"
    print(f"  {label}: {decode(snap)[ANS_START:]}")


## 5. Experimental Observations

First visualize a five-step generation. Rows are decoding times, columns are answer positions, and a filled square represents an unrevealed mask.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.2))
grid = np.zeros((len(snaps), ANS_LEN))
for r, snap in enumerate(snaps):
    for c, ch in enumerate(decode(snap)[ANS_START:]):
        grid[r, c] = 0.0 if ch == "█" else 1.0
ax.imshow(grid, cmap="Blues", aspect="auto", vmin=0, vmax=1)
for r, snap in enumerate(snaps):
    for c, ch in enumerate(decode(snap)[ANS_START:]):
        ax.text(c, r, ch, ha="center", va="center", fontsize=10,
                color="gray" if ch == "█" else "black")
ax.set_yticks(range(len(snaps)))
ax.set_yticklabels(["init"] + [f"step {i + 1}" for i in range(len(snaps) - 1)])
ax.set_xlabel("answer position")
ax.set_title("Masked diffusion decoding: positions get revealed over steps")
plt.show()
print("Key observation: easy blank positions appear first, while digits are revealed gradually by confidence.")
print("Every step writes the whole sequence together rather than one character at a time from left to right.")


Train an autoregressive baseline with the same TinyTransformer and `bidirectional=False`. It uses next-token prediction and generates left to right with one forward pass per token.


In [ ]:
def train_ar(model, steps=3000, bs=128, lr=1e-3):
    """Train an autoregressive baseline with next-Token Cross-Entropy at every position."""
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    hist = []
    for step in range(steps):
        batch = data[torch.randint(0, len(data), (bs,))].to(device)
        logits = model(batch[:, :-1])
        loss = F.cross_entropy(
            logits.reshape(-1, VOCAB), batch[:, 1:].reshape(-1)
        )
        opt.zero_grad()
        loss.backward()
        opt.step()
        if (step + 1) % 500 == 0:
            print(f"  step {step + 1:5d}  loss {loss.item():.4f}")
            hist.append((step + 1, loss.item()))
    return hist


@torch.no_grad()
def ar_generate(model, prompt_ids):
    """Autoregressive generation: one forward pass for each next Token, ANS_LEN passes total."""
    model.eval()
    seq = list(prompt_ids)
    for _ in range(ANS_LEN):
        ids = torch.tensor([seq], device=device)
        seq.append(int(model(ids)[0, -1].argmax()))
    return seq


In [ ]:
torch.manual_seed(42)
ar_model = TinyTransformer(bidirectional=False).to(device)
t0 = time.time()
ar_hist = train_ar(ar_model, steps=3000)
print(f"Training complete in {time.time() - t0:.0f} seconds")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.4))
dx, dy = zip(*diff_hist)
ax_, ay = zip(*ar_hist)
ax.plot(dx, dy, "o-", ms=4, label="masked diffusion (loss on masked positions)")
ax.plot(ax_, ay, "s-", ms=4, label="autoregressive (loss on all positions)")
ax.set_xlabel("training step")
ax.set_ylabel("cross-entropy loss")
ax.set_title("Both objectives converge")
ax.legend()
plt.show()
print("Key observation: convergence means each loss curve levels off; their magnitudes are not directly comparable because")
print("          diffusion loss covers only masked positions, naturally emphasizing difficult high-noise cases")


Now measure the **quality–step trade-off**. Generate the same 200 test samples with 1, 2, 3, 5, 8, and 15 diffusion steps, and compare exact-match accuracy with the 15-pass autoregressive baseline.


In [ ]:
def acc_diffusion(n_steps, n=200):
    ok = 0
    for s in tests[:n]:
        gen = diffuse_generate(diff_model, encode(s[:ANS_START]), n_steps)
        ok += (decode(gen) == s)
    return ok / n


def acc_ar(n=200):
    ok = 0
    for s in tests[:n]:
        gen = ar_generate(ar_model, encode(s[:ANS_START]))
        ok += (decode(gen) == s)
    return ok / n


step_list = [1, 2, 3, 5, 8, 15]
diff_accs = [acc_diffusion(ns) for ns in step_list]
ar_acc = acc_ar()

for ns, a in zip(step_list, diff_accs):
    print(f"diffusion {ns:2d} steps: exact-sequence accuracy {a:.1%}")
print(f"autoregressive {ANS_LEN} steps: exact-sequence accuracy {ar_acc:.1%}")
print()
print(f"Key observation: {step_list[0]} steps gives only {diff_accs[0]:.0%}, "
      f"while {step_list[-1]} steps reaches {diff_accs[-1]:.0%}; step count is the quality knob")


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))

ax1.plot(step_list, diff_accs, "o-", label="masked diffusion")
ax1.axhline(ar_acc, color="tomato", ls="--",
            label=f"autoregressive ({ANS_LEN} fwd passes)")
ax1.set_xlabel("number of diffusion steps")
ax1.set_ylabel("exact match accuracy")
ax1.set_title("Quality vs number of steps")
ax1.set_ylim(0, 1.05)
ax1.legend()

labels = ["AR"] + [f"{ns}-step" for ns in step_list]
fwd = [ANS_LEN] + step_list
ax2.bar(labels, fwd, color=["tomato"] + ["steelblue"] * len(step_list))
ax2.set_ylabel("forward passes per generation")
ax2.set_title("Cost: forward passes")
plt.tight_layout()
plt.show()

Three readings matter; exact percentages vary with training randomness:

- **One step is substantially worse.** A limited model cannot complete all reasoning in one pass and gets no chance to correct mistakes.
- **Step count is a quality control.** More steps generally improve exact match, allowing inference-time exchange of speed for quality.
- **Equal passes may still trail AR.** Even at 15 steps, the diffusion model may remain below the autoregressive baseline, matching the broader observation that diffusion LMs approach but do not consistently match equal-scale AR quality.

| | Autoregressive | One-step NAR | Masked Diffusion |
|:---|:---|:---|:---|
| Forward passes | $L$ serial | 1 | adjustable $k$ |
| Coordination | Left-to-right conditioning | None | Global rereading each step |
| Quality ceiling | Highest in this experiment | Lowest | Between NAR and AR |
| Unique control | None | None | Continuous speed–quality knob |


## 6. Differences from Production Systems

**Real models.** LLaDA (2025, 8B) continues training from a LLaMA-style initialization and approaches equal-scale AR models on several benchmarks. It uses a likelihood-weighted ELBO across noise levels and injects a timestep embedding, both omitted here.

**Remasking strategies:** random reveal, confidence-first reveal, semi-autoregressive blocks, and low-confidence remasking that allows already revealed positions to become masked again. The last enables revision, while our toy decoder never changes a committed token.

**Inference systems change.** Standard KV Cache no longer applies because bidirectional Attention recomputes the full sequence every step. Scheduling ideas remain useful, but bottlenecks differ from autoregressive serving.

**Current position.** Systems such as Mercury and Gemini Diffusion demonstrate commercial interest, with strongest speed advantages often reported on lower-entropy outputs such as code. Public quality commonly remains below similarly sized AR models, and most scaling, alignment, and serving infrastructure still targets AR. Hybrid AR–diffusion systems may bridge the transition.

Diffusion matters because it changes the generation paradigm rather than only optimizing the serial next-token loop.


## Summary

- AR needs $L$ ordered passes for $L$ tokens; speculation mitigates but retains the framework.
- One-step NAR fails when independently plausible positions form an invalid combination.
- Masked Diffusion uses `[MASK]` as noise and a bidirectional model for denoising over mask ratios from 0–100%.
- Each round predicts masks, reveals top-$k$ confidence positions, and postpones the rest; $k=\lceil m/s\rceil$ guarantees completion.
- Step count trades speed for quality, while equal-step quality may still trail AR.
- Production references include LLaDA, Mercury, and Gemini Diffusion; remasking and the loss of AR KV Cache reshape serving.

For further study, LLaDA's appendix derives the ELBO. Implementing low-confidence remasking in `diffuse_generate` is a strong comprehension exercise.


## Exercises

> You may ask AI for hints, decomposition, or direction checks, but avoid asking it to complete the exercises.

All exercises use the trained model and Section 4 code; no retraining is required.

1. **Sample the forward-process mask.** Hint: one `torch.rand` call and a comparison independently mask positions.
2. **Calculate reveals per step.** Hint: divide 15 positions across four steps and identify where floor division would fail.
3. **Compare random and confidence reveal.** Hint: `torch.randperm` provides a uniform random ordering.


In [ ]:
# Exercise 1: implement mask sampling for the forward process
# Independently mask each position with probability t, the definition of noise in masked diffusion

def sample_mask(seq_len, t, gen):
    """Return a Boolean tensor where True means replace this position with [MASK].
    seq_len: sequence length; t: noise level from 0 to 1; gen: generator for reproducibility.
    """
    rand = torch.rand(seq_len, generator=gen)
    mask = rand < t          # fill in one comparison operator
    return mask.bool()

g = torch.Generator().manual_seed(0)
m = sample_mask(1000, 0.3, g)
assert m.dtype == torch.bool, "mask should be a Boolean tensor"
assert 200 <= int(m.sum()) <= 400, \
    f"At t=0.3 and length 1,000, expect about 300 True values; you got {int(m.sum())}"
assert sample_mask(1000, 1.0, torch.Generator().manual_seed(1)).all(), \
    "At t=1.0 every position should be masked"

print("✅ Exercise 1 passed: you can sample sequence noise.")
print("   t is the masking ratio: larger t means stronger noise; at t=1 the whole sequence becomes [MASK].")


In [ ]:
# Exercise 2: how many positions should each step reveal?
# Too small will not finish; too large wastes steps

def reveal_count(remaining, steps_left):
    """remaining: [MASK] positions left; steps_left: steps remaining, including this one.
    Return the number of positions to reveal in this step.
    """
    return math.___(remaining / steps_left)   # fill in: ceil or floor?

# Reveal 15 positions in four steps: 4, 4, 4, 3
assert reveal_count(15, 4) == 4
assert reveal_count(11, 3) == 4
assert reveal_count(3, 1) == 3

print("✅ Exercise 2 passed: rounding upward guarantees completion exactly on time.")
print("   With floor, 15//4=3 reveals only 12 positions in four steps and leaves three without a step.")


In [ ]:
# Exercise 3: how much quality is lost by replacing confidence reveal with random reveal?

@torch.no_grad()
def diffuse_generate_random(model, prompt_ids, n_steps, seed=7):
    """Use the same logic as diffuse_generate, but randomly reveal k positions per step."""
    gen = torch.Generator().manual_seed(seed)
    model.eval()
    seq = list(prompt_ids) + [MASK_ID] * ANS_LEN
    masked = list(range(ANS_START, SEQ_LEN))
    for step_i in range(n_steps):
        ids = torch.tensor([seq], device=device)
        logits = model(ids)[0]
        pos = torch.tensor(masked, device=device)
        probs = F.softmax(logits[pos], dim=-1)
        _, pick = probs.max(dim=-1)
        k = math.ceil(len(masked) / (n_steps - step_i))
        top = torch.___(len(masked), generator=gen)[:k].tolist()  # fill in
        for j in top:
            seq[masked[j]] = pick[j].item()
        chosen = set(top)
        masked = [masked[j] for j in range(len(masked)) if j not in chosen]
    return seq

n_eval, n_steps = 100, 3
ok_r = ok_c = 0
for s in tests[:n_eval]:
    prompt = encode(s[:ANS_START])
    ok_r += (decode(diffuse_generate_random(diff_model, prompt, n_steps)) == s)
    ok_c += (decode(diffuse_generate(diff_model, prompt, n_steps)) == s)
acc_random, acc_conf = ok_r / n_eval, ok_c / n_eval
print(f"random reveal      3 steps: accuracy {acc_random:.0%}")
print(f"confidence reveal  3 steps: accuracy {acc_conf:.0%}")
assert acc_random <= acc_conf + 0.15, \
    "Random reveal usually does not outperform confidence-first, allowing for small random variation"

print("Exercise 3 passed: the reveal order itself carries information")
print("   Confidence resolves easy decisions first and leaves hard positions until the final round with the fullest context")
print("   The core remasking question is how to allocate the same number of forward passes most effectively")


## References

- Ho et al., 2020, *Denoising Diffusion Probabilistic Models* — foundational image diffusion work
- Gu et al., 2018, *Non-Autoregressive Neural Machine Translation* — early systematic study of NAR combination errors
- Nie et al., 2025, *Large Language Diffusion Models* — LLaDA, the main production-scale reference
- Official 2025 releases for Inception Labs Mercury and Google Gemini Diffusion
- Related Stanford CME295 (Autumn 2025) Lecture 9 and CS336 (Spring 2025) Lecture 10 notes
